# 09 — Disparadores externos y sincronización

Este notebook cubre todos los mecanismos disponibles para coordinar Python e IviumSoft:

- **Parámetro de estado** — entero compartido que tanto Python como los scripts Batch de IviumSoft pueden leer y escribir
- **Disparo de salida digital** — Python controla una línea hardware; el Batch de IviumSoft se detiene hasta que detecta el flanco
- **Disparo de salida analógica** — Python eleva un voltaje DAC por encima de un umbral; el Batch de IviumSoft se detiene hasta que se alcanza el nivel
- **IviumSoft → Python** — el Batch de IviumSoft lanza un script Python vía `ExecuteProgram` y espera a que termine
- **Transmisión multi-instancia** — enviar un disparo a todas las instancias activas de IviumSoft simultáneamente

### Cuándo usar cada mecanismo

| Mecanismo | Mejor para |
|-----------|----------|
| Parámetro de estado | Coordinación puramente software — sin cableado extra, funciona sobre cualquier conexión |
| Disparo digital | Flanco duro en tiempo real: latencia sub-ms, también funciona con instrumentos externos |
| Disparo analógico | Inicio basado en umbral: nivel de voltaje de un sensor o DAC |
| ExecuteProgram | IviumSoft conduce Python (post-procesamiento, notificaciones, generación de archivos) |

---

> ⚠️ **No verificado en producción.** Los patrones en este notebook se derivan de la referencia de la DLL de IviumSoft y el manual. Ninguno de ellos ha sido verificado en hardware real en un entorno de producción. Los valores de temporización, los detalles de configuración del Batch y el comportamiento en casos límite pueden diferir de lo descrito aquí.
>
> **Se necesita retroalimentación:** Si pruebas alguno de estos patrones y encuentras que algo se comporta de manera diferente — temporización incorrecta, disparos perdidos, configuración incorrecta del Batch, o cualquier cosa inesperada — por favor abre una issue en https://github.com/Gnpd/pyvium/issues con tu versión de IviumSoft y una descripción de lo que observaste. Tu experiencia en el mundo real es lo que hará que este notebook sea confiable.

---

### Requisitos previos
- IviumSoft instalado y en ejecución
- Driver abierto (ver `01_getting_started.ipynb`)
- Para disparos digitales/analógicos: método Batch de IviumSoft configurado con el `DirectCommand` correspondiente
- Referencia de manejo de errores: `01_getting_started.ipynb` — Sección 4

In [ ]:
import time
from pyvium import Pyvium

Pyvium.open_driver()
print(f"Active instances: {Pyvium.get_active_iviumsoft_instances()}")

## 1. El parámetro de estado

`set_status_par(value)` / `get_status_par()` exponen un entero global compartido entre Python e IviumSoft. Es **por instancia** — cada ventana de IviumSoft mantiene su propio valor independiente.

**Datos clave del manual de IviumSoft:**
- Inicializado a **`-1`** al arrancar el PC
- **Retiene su último valor establecido** hasta que el PC se reinicia — no se restablece entre mediciones o ciclos de apertura/cierre del driver
- Tanto Python (vía pyvium) como los scripts Batch de IviumSoft pueden leerlo y escribirlo
- La función DLL es `IV_setstatuspar` / `IV_getstatuspar`

### Convención recomendada

| Valor | Quién lo establece | Significado |
|-------|------------|----------|
| `-1` | Valor predeterminado al arrancar el PC | Sin inicializar |
| `0` | Python | Restablecer / inactivo |
| `1` | Python | "Listo — IviumSoft puede iniciar" |
| `2` | Batch de IviumSoft | "Medición en ejecución" |
| `3` | Batch de IviumSoft | "Medición completada" |
| `-99` | Cualquiera | Cancelar / error |

> **Siempre restablecer a `0` explícitamente** al inicio de cada flujo de trabajo. Debido a que el valor persiste hasta el reinicio del PC, un valor residual de una sesión anterior puede hacer que el `WaitForStatusPar` del Batch de IviumSoft se dispare inmediatamente y omita la espera prevista.

In [ ]:
Pyvium.select_iviumsoft_instance(1)

# Siempre restablecer antes de un nuevo flujo de trabajo
Pyvium.set_status_par(0)
print(f"Reset to 0")

# Leerlo de vuelta
val = Pyvium.get_status_par()
print(f"Current value: {val}")

## 2. Python → IviumSoft: Señalar a IviumSoft que inicie

Patrón: Python hace su trabajo de preparación, luego establece el parámetro de estado a `1`. Un script Batch de IviumSoft que ya está en ejecución se pausa en una línea `WaitForStatusPar` hasta que ve `1`, luego continúa.

**Configuración del Batch de IviumSoft:**
```
DirectCommand:
  WaitForStatusPar  ✓   Value = 1   Timeout = 300 s
```

**Flujo:**
```
Python:    set_status_par(0)        ← restablecer
Python:    ... preparar experimento ...
Python:    set_status_par(1)        ← disparar
IviumSoft: WaitForStatusPar termina ← Batch continúa
```

In [ ]:
Pyvium.select_iviumsoft_instance(1)
Pyvium.set_status_par(0)   # restablecer
print("Status par reset to 0 (idle)")

# ... tu trabajo de preparación aquí ...
time.sleep(0.5)            # simula el tiempo de preparación

Pyvium.set_status_par(1)   # disparar el trigger
print("Triggered IviumSoft (status par = 1)")

## 3. IviumSoft → Python: Esperar a que IviumSoft termine

Patrón: Python sondea `get_status_par()` en un bucle, esperando que IviumSoft establezca un valor específico cuando termina. El lado Batch de IviumSoft usa un DirectCommand `SetStatusPar` para escribir el valor.

**Configuración del Batch de IviumSoft:**
```
DirectCommand:
  SetStatusPar  ✓   Value = 3     ← escrito cuando la medición termina
```

**Flujo:**
```
IviumSoft: ... ejecuta experimento ...
IviumSoft: SetStatusPar = 3        ← señala a Python
Python:    get_status_par() → 3    ← bucle de sondeo termina
Python:    collect_data()
```

In [ ]:
def wait_for_status(instance: int, target: int, timeout_s: float = 120.0) -> bool:
    """Bloquear hasta que el parámetro de estado de la instancia dada alcance target, o tiempo de espera."""
    Pyvium.select_iviumsoft_instance(instance)
    t0 = time.time()
    while True:
        current = Pyvium.get_status_par()
        if current == target:
            return True
        if time.time() - t0 > timeout_s:
            print(f"Timeout: instance {instance} never reached status={target}")
            return False
        time.sleep(0.2)

print("wait_for_status() defined")

# Esperar hasta 60 s para que IviumSoft establezca el parámetro de estado a 3 ("listo")
# done = wait_for_status(instance=1, target=3, timeout_s=60.0)
# if done:
#     print("IviumSoft finished — collecting data")

## 4. El contrato bidireccional completo

Combinar las secciones 2 y 3 produce un handshake completo: Python señala a IviumSoft que inicie, IviumSoft señala a Python cuando termina.

| Paso | Lado Python | Lado Batch IviumSoft |
|------|-------------|----------------------|
| 1 | `set_status_par(0)` — restablecer | — |
| 2 | (preparar) | `WaitForStatusPar` (value=1, timeout=300s) — espera |
| 3 | `set_status_par(1)` — ir | `WaitForStatusPar` termina, Batch continúa |
| 4 | `wait_for_status(1, target=3)` — sondea | (ejecuta experimento) → `SetStatusPar` (value=3) |
| 5 | El sondeo termina, recopilar datos | — |

> **Tiempo de espera en ambos lados:** Siempre configura un tiempo de espera en `WaitForStatusPar` (lado IviumSoft) y en `wait_for_status()` (lado Python). Una señal perdida en cualquier lado nunca debe bloquear permanentemente la secuencia.

In [ ]:
def run_with_handshake(instance: int, timeout_s: float = 120.0):
    Pyvium.select_iviumsoft_instance(instance)

    # Paso 1: restablecer
    Pyvium.set_status_par(0)
    print(f"[inst {instance}] Reset — status=0")

    # Pasos 2-3: señalar a IviumSoft que inicie
    Pyvium.set_status_par(1)
    print(f"[inst {instance}] Triggered — status=1")

    # Paso 4: esperar a que IviumSoft reporte listo (status=3)
    done = wait_for_status(instance, target=3, timeout_s=timeout_s)
    if done:
        print(f"[inst {instance}] IviumSoft done — status=3")
    return done

print("run_with_handshake() defined")
# run_with_handshake(instance=1, timeout_s=60.0)

## 5. Disparo de salida digital

Python controla una línea de salida digital cableada a la entrada digital externa del CompactStat. El Batch de IviumSoft se detiene en `WaitForDigIn1` hasta que detecta el flanco.

**Configuración del Batch de IviumSoft:**
```
DirectCommand:
  WaitForDigIn1  ✓   WaitForHi = ✓   Timeout = 60 s
```

**Configuración hardware:**
```
Puerto externo del CompactStat
  Digital Output 0  ──────────────────────  Digital Input 1
                           (cable corto)
```

`set_digital_output(bitmask)` controla todas las líneas de salida digital simultáneamente. Cada bit corresponde a una línea:
```
0b00000001  →  línea 0 en ALTO
0b00000100  →  línea 2 en ALTO
0b00000000  →  todas las líneas en BAJO
```

> **Temporización:** IviumSoft sondea `DigIn1` aproximadamente cada 10–50 ms. Un pulso más corto que ~50 ms puede ser perdido. Mantén la línea en ALTO durante al menos 100 ms antes de liberarla.

In [ ]:
TRIGGER_LINE = 0   # línea de salida digital cableada a DigIn1 del dispositivo
HOLD_TIME_S  = 0.1 # mantener en ALTO durante 100 ms — suficiente para que IviumSoft detecte

def trigger_digital(line: int = TRIGGER_LINE, hold_s: float = HOLD_TIME_S):
    """Afirmar una línea de salida digital en ALTO, mantener, luego liberar."""
    Pyvium.set_digital_output(1 << line)
    print(f"Digital line {line} → HIGH")
    time.sleep(hold_s)
    Pyvium.set_digital_output(0)
    print(f"Digital line {line} → LOW (released)")

def read_digital_inputs() -> dict:
    """Devolver el estado actual de todas las líneas de entrada digital como un dict."""
    bitmask = Pyvium.get_digital_input()
    return {line: bool(bitmask & (1 << line)) for line in range(8)}

print("Digital trigger helpers defined")

# Leer el estado actual de la entrada digital
inputs = read_digital_inputs()
for line, state in inputs.items():
    print(f"  DigIn {line}: {'HIGH' if state else 'LOW'}")

# Disparar el trigger
# trigger_digital()

## 6. Disparo de salida analógica

Python eleva el canal DAC 0 por encima de un voltaje umbral. El Batch de IviumSoft se detiene en `WaitForAn1` hasta que la Entrada Analógica 1 supera el umbral configurado.

**Configuración del Batch de IviumSoft:**
```
DirectCommand:
  WaitForAn1  ✓   UntilAn1 > 2.5 V   Timeout = 60 s
```

**Configuración hardware:**
```
Puerto externo del CompactStat
  DAC channel 0  ──────────────────────  Analog Input 1
                        (cable corto)
```

**Elegir el umbral:**
- Establecer el umbral de IviumSoft a un valor que esté claramente por encima del voltaje DAC en reposo y por debajo del voltaje de disparo
- Ejemplo: reposo = 0 V, disparo = 3.0 V, umbral IviumSoft = 2.5 V
- Evita umbrales cerca del nivel de reposo — el ruido del DAC podría causar disparos falsos

> **Rango de salida del DAC:** típicamente ±10 V. Consulta la especificación de tu dispositivo antes de superar ±5 V.

In [ ]:
TRIGGER_VOLTAGE = 3.0   # V — debe superar el umbral del Batch de IviumSoft
IDLE_VOLTAGE    = 0.0   # V — nivel seguro cuando no se está disparando
DAC_CHANNEL     = 0     # canal DAC cableado a Analog Input 1
HOLD_TIME_S     = 0.5   # mantener por encima del umbral el tiempo suficiente para que IviumSoft detecte

def trigger_analog(
    channel: int = DAC_CHANNEL,
    trigger_v: float = TRIGGER_VOLTAGE,
    idle_v: float = IDLE_VOLTAGE,
    hold_s: float = HOLD_TIME_S,
):
    """Elevar el DAC a trigger_v, mantener, luego volver a idle_v."""
    Pyvium.set_dac(channel, trigger_v)
    print(f"DAC {channel} → {trigger_v} V (trigger)")
    time.sleep(hold_s)
    Pyvium.set_dac(channel, idle_v)
    print(f"DAC {channel} → {idle_v} V (idle)")

def read_analog_inputs(n_channels: int = 4) -> dict:
    """Leer los primeros n_channels de entrada ADC y devolver como un dict."""
    return {ch: Pyvium.get_adc(ch) for ch in range(n_channels)}

print("Analog trigger helpers defined")

# Leer valores ADC actuales
adc = read_analog_inputs()
for ch, v in adc.items():
    print(f"  ADC {ch}: {v:.4f} V")

# Disparar el trigger
# trigger_analog()

## 7. IviumSoft lanza Python (`ExecuteProgram`)

La dirección inversa: un script Batch de IviumSoft lanza un script Python en un punto específico de su secuencia y opcionalmente espera a que termine antes de continuar.

**Configuración del Batch de IviumSoft:**
```
DirectCommand:
  ExecuteProgram     ✓
  Program Name:      python.exe
  Path:              C:\Users\...\Scripts\
  Command Line:      C:\ruta\a\mi_script.py --output C:\datos\resultado.csv
  WaitUntilFinished: ✓
```

**Secuencia típica:**
```
[Línea Batch N]    ExecuteMethod           ← ejecutar experimento electroquímico
[Línea Batch N+1]  ExecuteProgram          ← lanzar Python, esperar a que termine
[Línea Batch N+2]  Volver / siguiente barrido  ← resultado de Python ya en disco
```

**Casos de uso:**
- Post-procesar datos inmediatamente después de cada barrido dentro de un bucle Batch
- Enviar una notificación (correo electrónico, Slack) cuando termina un experimento largo
- Leer el resultado de la medición y escribir un archivo de parámetros para la siguiente línea Batch
- Ejecutar control de calidad — cancelar el Batch si el resultado está fuera de rango

**Convención de código de salida:**
IviumSoft no inspecciona el código de salida de Python — solo espera a que el proceso termine. Para señalar de vuelta a IviumSoft desde Python, escribe un archivo o usa `set_status_par()` antes de salir.

In [ ]:
# Plantilla para un script lanzado por IviumSoft vía ExecuteProgram
# Guardar como un archivo .py independiente; IviumSoft lo llamará y esperará a que termine.

TEMPLATE = '''
import sys
import argparse
from pyvium import Pyvium

parser = argparse.ArgumentParser()
parser.add_argument("--output", required=True)
args = parser.parse_args()

Pyvium.open_driver()
Pyvium.select_iviumsoft_instance(1)

n = Pyvium.get_available_data_points_number()
rows = [Pyvium.get_data_point(i) for i in range(1, n + 1)]

import csv
with open(args.output, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["E (V)", "I (A)", "aux"])
    writer.writerows(rows)

# Opcionalmente señalar de vuelta a IviumSoft vía parámetro de estado
Pyvium.set_status_par(10)   # "Post-procesamiento Python listo"

Pyvium.close_driver()
sys.exit(0)
'''

print("ExecuteProgram script template:")
print(TEMPLATE)

## 8. Transmisión multi-instancia

Para disparar todas las instancias activas de IviumSoft casi simultáneamente, itera por ellas y establece el parámetro de estado en cada una. El bucle en sí toma solo unos pocos milisegundos, por lo que todas las instancias reciben la señal dentro de un ciclo de sondeo de IviumSoft.

Este es el patrón estándar para iniciar experimentos paralelos en múltiples dispositivos al mismo tiempo.

In [ ]:
def broadcast_status(value: int):
    """Establecer el parámetro de estado en todas las instancias activas de IviumSoft."""
    instances = Pyvium.get_active_iviumsoft_instances()
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.set_status_par(value)
    print(f"Broadcast status={value} to instances {instances}")
    return instances

def wait_all_status(instances: list, target: int, timeout_s: float = 120.0) -> bool:
    """Esperar hasta que el parámetro de estado de cada instancia sea igual a target, o tiempo de espera."""
    remaining = set(instances)
    t0 = time.time()
    while remaining:
        finished = set()
        for inst in remaining:
            Pyvium.select_iviumsoft_instance(inst)
            if Pyvium.get_status_par() == target:
                finished.add(inst)
                print(f"  Instance {inst}: status={target}")
        remaining -= finished
        if remaining and time.time() - t0 > timeout_s:
            print(f"Timeout — still waiting on instances {remaining}")
            return False
        if remaining:
            time.sleep(0.2)
    return True

print("Broadcast helpers defined")

# Restablecer todos, luego disparar todos
instances = broadcast_status(0)   # restablecer
time.sleep(0.5)
broadcast_status(1)               # ir

## 9. Elegir entre tipos de disparo

### Parámetro de estado vs disparadores hardware

| Criterio | Parámetro de estado | Disparo digital | Disparo analógico |
|-----------|-----------------|-----------------|--------------------|
| Cableado requerido | Ninguno | Sí (1 cable) | Sí (1 cable) |
| Latencia | ~200 ms (ciclo de sondeo) | ~50 ms (sondeo IviumSoft) | ~50 ms (sondeo IviumSoft) |
| Funciona a través de red | Sí | No | No |
| Señal reversible | Sí (establecer de nuevo) | Sí (liberar línea) | Sí (bajar DAC) |
| Lado Batch IviumSoft | `WaitForStatusPar` | `WaitForDigIn1` | `WaitForAn1` |
| También puede recibir señal | Sí (`get_status_par`) | Sí (`get_digital_input`) | Sí (`get_adc`) |

### El tiempo de espera importa en ambos lados

Los tres DirectCommands `WaitFor...` de IviumSoft aceptan un ajuste de **Timeout**. Si el disparo nunca llega, el Batch continúa después de que expira el tiempo de espera en lugar de quedarse colgado permanentemente. Siempre configura un tiempo de espera que sea:
- Lo suficientemente largo para cubrir el tiempo de preparación de Python en el peor caso
- Lo suficientemente corto para fallar rápido si algo salió mal

Haz coincidir el `timeout_s` del lado Python en tu llamada `wait_for_status()` con el mismo presupuesto de tiempo.

## 10. Plantilla completa de flujo de trabajo con disparadores

Un ejemplo autocontenido que:
1. Restablece todas las instancias
2. Dispara un disparo software a todas las instancias simultáneamente
3. Espera a que todas las instancias informen que han terminado
4. Recopila datos de todas las instancias

In [ ]:
import time
import pandas as pd
from pyvium import Pyvium

STATUS_IDLE    = 0
STATUS_GO      = 1
STATUS_RUNNING = 2
STATUS_DONE    = 3
STATUS_ABORT   = -99
TRIGGER_TIMEOUT_S = 120.0

Pyvium.open_driver()
instances = Pyvium.get_active_iviumsoft_instances()
print(f"Devices: {instances}")

# Paso 1: restablecer todos
broadcast_status(STATUS_IDLE)

# Paso 2: conectar todos
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.connect_device()
    print(f"  Instance {inst}: connected")

# Paso 3: disparar todos simultáneamente
# El Batch de IviumSoft en cada instancia debe estar configurado con WaitForStatusPar (value=1)
broadcast_status(STATUS_GO)

# Paso 4: esperar a que todos terminen (IviumSoft establece status=3 vía SetStatusPar)
all_done = wait_all_status(instances, target=STATUS_DONE, timeout_s=TRIGGER_TIMEOUT_S)

if not all_done:
    broadcast_status(STATUS_ABORT)
    print("Aborted — not all instances finished in time")
else:
    # Paso 5: recopilar datos de todos
    results = {}
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        n = Pyvium.get_available_data_points_number()
        rows = [Pyvium.get_data_point(i) for i in range(1, n + 1)]
        results[inst] = pd.DataFrame(rows, columns=["E (V)", "I (A)", "aux"])
        print(f"  Instance {inst}: {len(rows)} points")

Pyvium.close_driver()
print("Done")

## Limpieza

In [ ]:
# Restablecer todas las salidas digitales y analógicas a un estado seguro
Pyvium.set_digital_output(0)
Pyvium.set_dac(0, 0.0)
Pyvium.set_dac(1, 0.0)
Pyvium.close_driver()
print("Outputs reset, driver closed")

---

## Resumen

### Parámetro de estado

| Tarea | Python | Batch de IviumSoft |
|------|--------|-------------------|
| Restablecer | `set_status_par(0)` | — |
| Señalar a IviumSoft que inicie | `set_status_par(1)` | `WaitForStatusPar` (value=1, timeout) |
| Esperar a que IviumSoft termine | `get_status_par()` en bucle de sondeo | `SetStatusPar` (value=3) |
| Transmitir a todas las instancias | bucle `select_iviumsoft_instance(n)` + `set_status_par(v)` | — |
| Valor inicial al arrancar el PC | — | **-1** (retiene hasta reinicio del PC) |

### Disparadores hardware

| Tipo | Python envía | IviumSoft Batch espera en | Lectura de vuelta |
|------|-------------|--------------------------|------------------|
| Digital | `set_digital_output(bitmask)` | `WaitForDigIn1` (HI/LO, timeout) | `get_digital_input()` |
| Analógico | `set_dac(0, voltage)` | `WaitForAn1 > threshold` (timeout) | `get_adc(channel)` |

### Python conducido por IviumSoft

| Método | Batch de IviumSoft | Lado Python |
|--------|----------------|------------|
| Lanzar script | `ExecuteProgram` + `WaitUntilFinished` | archivo `.py` independiente, termina con `sys.exit(0)` |

### Lista de verificación de seguridad

- Siempre restablecer el parámetro de estado a `0` antes de un nuevo flujo de trabajo
- Siempre configurar un tiempo de espera en `WaitForStatusPar` / `WaitForDigIn1` / `WaitForAn1` en el lado de IviumSoft
- Siempre establecer un `timeout_s` coincidente en el lado Python del sondeo
- Restablecer el DAC a `0 V` y las salidas digitales a `0` en tu celda de limpieza
- Capturar `IllegalCommandError` al llamar a `abort_method()` de forma defensiva

---

## Notebooks relacionados

| Notebook | Tema |
|----------|------|
| `04_direct_mode_signals.ipynb` | Salida DAC, entrada ADC, referencia de E/S digital |
| `06_method_mode.ipynb` | Ejecutar y monitorear métodos, `abort_method()` |
| `08_batch_and_synchronization.ipynb` | Puntos de ajuste multi-dispositivo, mediciones paralelas, inicio de canal sincronizado |